In [1]:
# This file compares new implementation V2 of Algorithm 4 with the old implementations.



In [2]:
%%writefile Algo_4_old_implementation.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <algorithm>
#include <random>
#include <numeric>
#include <set>
#include <tuple>
#include <iomanip>
#include <cmath>
#include <chrono>
#include <limits>
#include <functional>

using namespace std;

using Matrix = vector<vector<double>>;
constexpr double INF = numeric_limits<double>::infinity();
using Edge = tuple<int, int, double>; // (u, v, weight)

// Helper function to compare two matrices
bool areMatricesEqual(const vector<vector<double>>& matrix1, const vector<vector<double>>& matrix2) {
    const double epsilon = 1e-6;
    int n = matrix1.size();
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            if (fabs(matrix1[i][j] - matrix2[i][j]) > epsilon) {
                cout << "Mismatch at (" << i << ", " << j << "): "
                     << "Matrix1 = " << matrix1[i][j] << ", Matrix2 = " << matrix2[i][j] << endl;
                return false;
            }
        }
    }
    return true;
}


// Prim's MST with min-heap optimization
vector<int> primMST(const Matrix& dist) {
    int V = dist.size();
    vector<double> key(V, INF);
    vector<int> parent(V, -1);
    vector<bool> inMST(V, false);

    key[0] = 0.0;
    priority_queue<pair<double, int>, vector<pair<double, int>>, greater<>> pq;
    pq.emplace(0.0, 0);

    while (!pq.empty()) {
        auto [k, u] = pq.top(); pq.pop();
        if (inMST[u]) continue;
        inMST[u] = true;

        for (int v = 0; v < V; ++v) {
            if (dist[u][v] && !inMST[v] && dist[u][v] < key[v]) {
                key[v] = dist[u][v];
                parent[v] = u;
                pq.emplace(key[v], v);
            }
        }
    }
    return parent;
}

vector<vector<double>> create_symmetric_distance_matrix(int N, int seed) {
    mt19937 gen(seed);

    // Generate doubles between 1.00 and 19999.00
    uniform_real_distribution<double> dist(1.0, 19999.0);

    vector<vector<double>> A(N, vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {
        for (int j = i + 1; j < N; ++j) {

            // Round to 2 decimal places
            double val = round(dist(gen) * 100.0) / 100.0;

            A[i][j] = A[j][i] = val;
        }
    }

    return A;
}



// MMJ matrix calculation using Algorithm 4 (Calculation and Copy)
vector<vector<double>> calculateMMJMatrixAlgo4(const vector<vector<double>>& distanceMatrix) {
    int n = distanceMatrix.size();
    vector<vector<double>> mmjMatrix(n, vector<double>(n, 0));

    // Construct MST

    auto parent = primMST(distanceMatrix);

    vector<Edge> mstEdges;
    for (int i = 1; i < n; ++i)
        mstEdges.emplace_back(min(i, parent[i]), max(i, parent[i]), distanceMatrix[i][parent[i]]);


    // Sort edges by weight in descending order
    sort(mstEdges.begin(), mstEdges.end(), [](const auto& a, const auto& b) {
        return get<2>(a) > get<2>(b);
    });

    set<int> tree1Nodes, tree2Nodes;
    vector<set<int>> adjacencyList(n);
    for (const auto& edge : mstEdges) {
        adjacencyList[get<0>(edge)].insert(get<1>(edge));
        adjacencyList[get<1>(edge)].insert(get<0>(edge));
    }

    for (const auto& edge : mstEdges) {
        int u = get<0>(edge), v = get<1>(edge);
        double weight = get<2>(edge);

        adjacencyList[u].erase(v);
        adjacencyList[v].erase(u);

        vector<bool> visited(n, false);
        tree1Nodes.clear();
        tree2Nodes.clear();

        function<void(int, set<int>&)> dfs = [&](int node, set<int>& tree) {
            visited[node] = true;
            tree.insert(node);
            for (int neighbor : adjacencyList[node]) {
                if (!visited[neighbor]) dfs(neighbor, tree);
            }
        };

        dfs(u, tree1Nodes);
        dfs(v, tree2Nodes);

        for (int p1 : tree1Nodes) {
            for (int p2 : tree2Nodes) {
                mmjMatrix[p1][p2] = mmjMatrix[p2][p1] = weight;
            }
        }
    }

    return mmjMatrix;
}

int main() {

    int n = 30000;
    int seed = 2835;

  cout << "Number of nodes: "
         << n << endl;

    auto distanceMatrix = create_symmetric_distance_matrix(n, seed);



    auto start = chrono::high_resolution_clock::now();
    auto mmjMatrixAlgo4 = calculateMMJMatrixAlgo4(distanceMatrix);
    auto end = chrono::high_resolution_clock::now();
    cout << "Time used (Algorithm 4) - old implementation: " << chrono::duration<double>(end - start).count() << " seconds" << endl;


    cout << "Print last 30 values of the first row of mmj matrix:\n";

    const auto& row = mmjMatrixAlgo4[0];
    for (size_t i = row.size() - 30; i < row.size(); ++i)
        cout << fixed << setprecision(2) << row[i] << " ";
    cout << "\n";

    return 0;
}



Overwriting Algo_4_old_implementation.cpp


In [3]:
%%writefile Algo_4_new_implementation.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <algorithm>
#include <random>
#include <numeric>
#include <set>
#include <tuple>
#include <iomanip>
#include <cmath>
#include <chrono>
#include <limits>
#include <stack>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;

const double INF = numeric_limits<double>::infinity();

// ============================================================
// GRAPH
// ============================================================

struct AdjEdge {
    int to;
    int id;
};

// ============================================================
// PRIM MST
// ============================================================

vector<int> primMST(const Matrix& dist) {

    int n = dist.size();

    vector<double> key(n, INF);
    vector<int> parent(n, -1);
    vector<char> inMST(n, 0);

    priority_queue<
        pair<double,int>,
        vector<pair<double,int>>,
        greater<>
    > pq;

    key[0] = 0.0;

    pq.emplace(0.0, 0);

    while (!pq.empty()) {

        auto [k, u] = pq.top();
        pq.pop();

        if (inMST[u])
            continue;

        inMST[u] = 1;

        const double* row = dist[u].data();

        for (int v = 0; v < n; ++v) {

            double w = row[v];

            if (w && !inMST[v] && w < key[v]) {

                key[v] = w;
                parent[v] = u;

                pq.emplace(w, v);
            }
        }
    }

    return parent;
}

// ============================================================
// BUILD GRAPH
// ============================================================

void buildGraph(
    int n,
    const vector<Edge>& edge_list,
    vector<vector<AdjEdge>>& graph)
{
    graph.assign(n, {});

    for (int i = 0; i < (int)edge_list.size(); ++i) {

        auto [u, v, w] = edge_list[i];

        graph[u].push_back({v, i});
        graph[v].push_back({u, i});
    }
}

// ============================================================
// FAST DFS
// ============================================================

inline void dfs_fast(
    int start,
    const vector<vector<AdjEdge>>& graph,
    const vector<char>& active,
    vector<int>& visited,
    int token,
    vector<int>& nodes)
{
    nodes.clear();

    stack<int> st;

    st.push(start);

    visited[start] = token;

    while (!st.empty()) {

        int u = st.top();
        st.pop();

        nodes.push_back(u);

        for (const auto& e : graph[u]) {

            if (!active[e.id])
                continue;

            int v = e.to;

            if (visited[v] != token) {

                visited[v] = token;

                st.push(v);
            }
        }
    }
}



Matrix cal_mmj_matrix_by_algo_4_new_implementation(const Matrix& distance_matrix)
{
    int n = distance_matrix.size();

    Matrix mmj_matrix(
        n,
        vector<double>(n, 0.0));

    auto parent = primMST(distance_matrix);

    vector<Edge> edge_list;

    edge_list.reserve(n - 1);

    for (int i = 1; i < n; ++i) {

        edge_list.emplace_back(
            min(i, parent[i]),
            max(i, parent[i]),
            distance_matrix[i][parent[i]]);
    }

    sort(
        edge_list.begin(),
        edge_list.end(),
        [](const Edge& a, const Edge& b) {

            return get<2>(a) > get<2>(b);
        });

    // ========================================================
    // EDGE ARRAYS
    // ========================================================

    vector<pair<int,int>> edge_nodes;

    vector<double> edge_weights;

    edge_nodes.reserve(n - 1);
    edge_weights.reserve(n - 1);

    for (const auto& [u, v, w] : edge_list) {

        edge_nodes.emplace_back(u, v);

        edge_weights.push_back(w);
    }

    // ========================================================
    // GRAPH
    // ========================================================

    vector<vector<AdjEdge>> graph;

    buildGraph(n, edge_list, graph);

    int num_edges = n - 1;

    // initially all active
    vector<char> active(num_edges, 1);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;


    for (int task = 0; task < n - 1; ++task)
    {
        active[task] = 0;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }

    return mmj_matrix;
}

// ============================================================
// DISTANCE MATRIX
// ============================================================

vector<vector<double>> createDistanceMatrix(
    int N,
    int seed)
{
    mt19937 gen(seed);

    uniform_real_distribution<double>
        dist(1.0, 19999.0);

    vector<vector<double>> A(
        N,
        vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {

        for (int j = i + 1; j < N; ++j) {

            double val =
                round(dist(gen) * 100.0) / 100.0;

            A[i][j] = val;
            A[j][i] = val;
        }
    }

    return A;
}


// ============================================================
// MAIN
// ============================================================

int main() {

    int N =30000;



    int random_seed = 2835;

    cout << "Number of nodes: "
         << N << endl;



    auto distanceMatrix = createDistanceMatrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();

    auto mmjMatrix = cal_mmj_matrix_by_algo_4_new_implementation(distanceMatrix);

    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);

    cout << "Time used (Algorithm 4) - new implementation:"
         << time_used
         << " seconds\n";

    cout << "Print last 30 values of the first row of mmj matrix:\n";

    const auto& row = mmjMatrix[0];

    for (size_t i = row.size() - 30; i < row.size(); ++i)
    {
        cout << fixed
             << setprecision(2)
             << row[i]
             << " ";
    }

    cout << "\n";

    return 0;
}


Overwriting Algo_4_new_implementation.cpp


In [4]:
%%writefile Algo_4_new_implementation_V2.cpp
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <limits>
#include <memory>
#include <queue>
#include <random>
#include <utility>
#include <vector>

using namespace std;

class Matrix {
public:
    explicit Matrix(int n) : n_(n), data_(new double[size_t(n) * n]) {}
    Matrix(Matrix&&) noexcept = default;
    Matrix& operator=(Matrix&&) noexcept = default;
    Matrix(const Matrix&) = delete;
    Matrix& operator=(const Matrix&) = delete;

    int size() const { return n_; }
    double* row(int i) { return data_.get() + size_t(i) * n_; }
    const double* row(int i) const { return data_.get() + size_t(i) * n_; }
    double& operator()(int i, int j) { return row(i)[j]; }
    double operator()(int i, int j) const { return row(i)[j]; }
    void release() { data_.reset(); n_ = 0; }

private:
    int n_;
    unique_ptr<double[]> data_;
};

struct Edge {
    int u, v;
    double weight;
};


vector<Edge> primMST(const Matrix& dist) {
    const int n = dist.size();
    vector<double> key(n, numeric_limits<double>::infinity());
    vector<int> parent(n, -1);
    vector<char> in_mst(n, 0);
    using Item = pair<double, int>;
    priority_queue<Item, vector<Item>, greater<Item>> pq;
    key[0] = 0.0;
    pq.emplace(0.0, 0);

    while (!pq.empty()) {
        const int u = pq.top().second;
        pq.pop();
        if (in_mst[u]) continue;
        in_mst[u] = 1;
        const double* drow = dist.row(u);
        for (int v = 0; v < n; ++v) {
            const double w = drow[v];
            if (!in_mst[v] && w < key[v]) {
                key[v] = w;
                parent[v] = u;
                pq.emplace(w, v);
            }
        }
    }

    vector<Edge> mst;
    mst.reserve(n - 1);
    for (int v = 1; v < n; ++v)
        mst.push_back({parent[v], v, dist(v, parent[v])});
    return mst;
}

// Kruskal reconstruction principle: when an edge of weight w joins two
// components, every cross-component pair has MMJ distance w.
// Each component is a singly linked list of original vertex IDs. Merging
// lists costs O(1), so no DFS or matrix permutation is required.
Matrix mmjFromMST(int n, vector<Edge>& mst) {
    sort(mst.begin(), mst.end(), [](const Edge& a, const Edge& b) {
        return a.weight < b.weight;
    });

    Matrix mmj(n);
    vector<int> parent(n), component_size(n, 1);
    vector<int> head(n), tail(n), next(n, -1);
    for (int v = 0; v < n; ++v) {
        parent[v] = head[v] = tail[v] = v;
        mmj(v, v) = 0.0;
    }

    auto find = [&](int x) {
        int root = x;
        while (parent[root] != root) root = parent[root];
        while (parent[x] != x) {
            const int old_parent = parent[x];
            parent[x] = root;
            x = old_parent;
        }
        return root;
    };

    for (const Edge& edge : mst) {
        int a = find(edge.u);
        int b = find(edge.v);
        if (component_size[a] < component_size[b]) swap(a, b);

        for (int u = head[a]; u != -1; u = next[u]) {
            double* row_u = mmj.row(u);
            for (int v = head[b]; v != -1; v = next[v]) {
                row_u[v] = edge.weight;
                mmj(v, u) = edge.weight;
            }
        }

        // Append b to a after filling: both lists were separate above.
        next[tail[a]] = head[b];
        tail[a] = tail[b];
        parent[b] = a;
        component_size[a] += component_size[b];
    }
    return mmj;
}

Matrix createDistanceMatrix(int n, int seed) {
    mt19937 gen(seed);
    uniform_real_distribution<double> dist(1.0, 19999.0);
    Matrix result(n);
    for (int i = 0; i < n; ++i) {
        result(i, i) = 0.0;
        for (int j = i + 1; j < n; ++j) {
            const double w = round(dist(gen) * 100.0) / 100.0;
            result(i, j) = w;
            result(j, i) = w;
        }
    }
    return result;
}

int main() {
    const int N = 30000;
    const int random_seed = 2835;
    cout << "Number of nodes: " << N << '\n';
    auto distance_matrix = createDistanceMatrix(N, random_seed);

    // One total timer, including Prim and the MMJ matrix calculation.
    const auto start = chrono::steady_clock::now();
    auto mst = primMST(distance_matrix);
    distance_matrix.release();
    auto mmj_matrix = mmjFromMST(N, mst);
    const auto end = chrono::steady_clock::now();

    cout << fixed << setprecision(3);
    cout << "Time used (Algorithm 4) - new implementation V2: "
         << chrono::duration<double>(end - start).count() << " seconds\n";
    cout << "Print last 30 values of the first row of mmj matrix:\n";
    for (int j = max(0, N - 30); j < N; ++j)
        cout << setprecision(2) << mmj_matrix(0, j) << ' ';
    cout << '\n';
    return 0;
}


Overwriting Algo_4_new_implementation_V2.cpp


In [5]:
!g++ -std=c++17 -O3 -march=native Algo_4_old_implementation.cpp  -o tt
!./tt

Number of nodes: 30000
Time used (Algorithm 4) - old implementation: 78.8868 seconds
Print last 30 values of the first row of mmj matrix:
3.15 2.76 1.92 3.62 1.92 1.92 1.92 1.92 1.92 2.20 3.11 2.00 1.93 1.92 1.96 2.39 1.92 2.54 2.53 2.27 1.97 1.92 2.21 3.00 1.92 1.95 1.92 2.75 1.97 1.92 


In [6]:
!g++ -std=c++17 -O3 -march=native Algo_4_new_implementation.cpp  -o tt
!./tt

Number of nodes: 30000
Time used (Algorithm 4) - new implementation:21.221 seconds
Print last 30 values of the first row of mmj matrix:
3.15 2.76 1.92 3.62 1.92 1.92 1.92 1.92 1.92 2.20 3.11 2.00 1.93 1.92 1.96 2.39 1.92 2.54 2.53 2.27 1.97 1.92 2.21 3.00 1.92 1.95 1.92 2.75 1.97 1.92 


In [7]:
!g++ -O3 -march=native -std=c++17  Algo_4_new_implementation_V2.cpp -o tt
!./tt

Number of nodes: 30000
Time used (Algorithm 4) - new implementation V2: 10.740 seconds
Print last 30 values of the first row of mmj matrix:
3.15 2.76 1.92 3.62 1.92 1.92 1.92 1.92 1.92 2.20 3.11 2.00 1.93 1.92 1.96 2.39 1.92 2.54 2.53 2.27 1.97 1.92 2.21 3.00 1.92 1.95 1.92 2.75 1.97 1.92 


In [8]:

# New implementation V2 is quite faster than the old implementations.

In [9]:
import platform
import psutil

# CPU information
print("CPU Information:")
print(f"Processor: {platform.processor()}")
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"CPU frequency: {psutil.cpu_freq().current:.2f} MHz")

# RAM information
ram = psutil.virtual_memory()

print("\nRAM Information:")
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")

CPU Information:
Processor: x86_64
Physical cores: 4
Logical cores: 8
CPU frequency: 2250.00 MHz

RAM Information:
Total RAM: 50.99 GB
